# Choosing per-well display ranges (vmin / vmax) for a channel


**Logic**
1. For each channel, pick **one pair of pixel percentiles** (low → `vmin`, high → `vmax`) that gives a good visualisation. Bright or strong stains need a lower high percentile; weak stains need a higher one.
2. Staining intensity varies from well to well, so a single absolute `vmin`/`vmax` would make some wells look dimmer or brighter than they are. Instead, **keep the percentiles constant and recompute `vmin`/`vmax` separately for each well**. This normalises the display per well, so crops can be compared across wells.
3. Percentiles are computed on pixels pooled from several large patches inside the central part of each well. This avoids edges and empty margins.
4. The printed `channel_info_<WELL> = [...]` lines are pasted into the per-channel grid script (`template_channel_grid.py` → `CHANNEL_INFO`).

Fill in every `<...>` placeholder before running.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scallops.io import read_image

## 1. Configuration

In [ ]:
# Wells and where their registered phenotype images live.
# Use the SAME image source here as in the grid script, so the percentiles are
# computed on the images that are actually displayed.
WELLS = ["<WELL_1>", "<WELL_2>"]
IMAGE_PATH = "<REGISTERED_PHENO_ZARR>/images/{well}"
IMAGE_PATH_OVERRIDES = {}  # e.g. {"<WELL>": "<OTHER_ZARR>/images/{well}"}

# Channel index -> name, in image order
CHANNEL_NAMES = ["<CH_0>", "<CH_1>"]

# Percentiles: one default pair, plus per-channel overrides chosen by eye.
# Keep these constant across wells.
DEFAULT_LOW_PCT = 0.0    # <set>, -> vmin
DEFAULT_HIGH_PCT = 99.5  # <set>, -> vmax
PCT_OVERRIDES = {
    # channel_idx: (low_pct, high_pct)
}

# Pixel sampling
PATCH_SIZE = 5000      # pixels per side
MARGIN_FRAC = 0.10     # skip this fraction of the image on every side

## 2. Helpers

In [ ]:
def image_path(well):
    return IMAGE_PATH_OVERRIDES.get(well, IMAGE_PATH).format(well=well)


def patch_origins(full_y, full_x):
    """Top-left corners of 5 patches (center + 4 quadrants) inside the central region."""
    my, mx = int(MARGIN_FRAC * full_y), int(MARGIN_FRAC * full_x)
    sy, sx = full_y - 2 * my, full_x - 2 * mx
    return [
        (my + sy // 2,     mx + sx // 2),      # center
        (my + sy // 4,     mx + sx // 4),      # upper-left
        (my + sy // 4,     mx + 3 * sx // 4),  # upper-right
        (my + 3 * sy // 4, mx + sx // 4),      # lower-left
        (my + 3 * sy // 4, mx + 3 * sx // 4),  # lower-right
    ]


def percentiles_for(c):
    return PCT_OVERRIDES.get(c, (DEFAULT_LOW_PCT, DEFAULT_HIGH_PCT))


def display_range(im, c, origins):
    """vmin/vmax for channel c from pixels pooled over the sampled patches."""
    pixels = np.concatenate([
        im.isel(c=c, y=slice(y, y + PATCH_SIZE), x=slice(x, x + PATCH_SIZE)).values.ravel()
        for y, x in origins
    ])
    lo, hi = percentiles_for(c)
    return np.percentile(pixels, lo), np.percentile(pixels, hi)

## 3. Compute vmin / vmax per well (percentiles held constant)

In [ ]:
rows = []
images = {}

for well in WELLS:
    im = read_image(image_path(well), dask=True).squeeze()
    images[well] = im
    origins = patch_origins(im.sizes["y"], im.sizes["x"])

    channel_info = []
    for c, name in enumerate(CHANNEL_NAMES):
        vmin, vmax = display_range(im, c, origins)
        lo, hi = percentiles_for(c)
        channel_info.append((c, int(vmin), int(vmax), name))
        rows.append(dict(well=well, channel=c, name=name, low_pct=lo, high_pct=hi,
                         vmin=int(vmin), vmax=int(vmax)))

    # Paste these lines into the grid script
    print(f"channel_info_{well} = {channel_info}")

df_ranges = pd.DataFrame(rows)
df_ranges.pivot(index="name", columns="well", values=["vmin", "vmax"])

## 4. Visual check

For one channel, show the central patch of every well with that well's own vmin/vmax.
If the percentiles are well chosen, staining patterns should look comparable across wells.
Change the percentiles in section 1 and re-run until they do.

In [ ]:
CHANNEL_TO_CHECK = 0
PREVIEW_SIZE = 2000

fig, axs = plt.subplots(1, len(WELLS), figsize=(5 * len(WELLS), 5))
axs = np.atleast_1d(axs)
for ax, well in zip(axs, WELLS):
    im = images[well]
    cy, cx = im.sizes["y"] // 2, im.sizes["x"] // 2
    crop = im.isel(c=CHANNEL_TO_CHECK,
                   y=slice(cy - PREVIEW_SIZE // 2, cy + PREVIEW_SIZE // 2),
                   x=slice(cx - PREVIEW_SIZE // 2, cx + PREVIEW_SIZE // 2))
    r = df_ranges.query("well == @well and channel == @CHANNEL_TO_CHECK").iloc[0]
    ax.imshow(crop, cmap="gray_r", vmin=r.vmin, vmax=r.vmax)  # same cmap as the grids
    ax.set_title(f"{well} | {r['name']} | {r.vmin}-{r.vmax}")
    ax.axis("off")
plt.tight_layout()
plt.show()
plt.close(fig)